In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler

In [31]:
df = pd.read_csv("A:\\iam_pikabyte\\CSV_Files&Pdf's\\titanic_data_updated.csv")
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,no,second,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,yes,first,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,no,third,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,yes,first,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [32]:
df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

X = df.drop(['Survived'], axis=1)
y=df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [33]:
imputer_transformer = ColumnTransformer(
    transformers=[
        ('age', SimpleImputer(missing_values=np.nan, strategy='mean'),['Age']),
        ('embarked',SimpleImputer(missing_values=np.nan, strategy='most_frequent'),['Embarked']),
        ('cabin', SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='Missing', add_indicator=True),['Cabin'])
    ],remainder = 'passthrough',
    verbose_feature_names_out = False
)

imputer_transformer.set_output(transform='pandas')

imputer_transformer.fit(X_train)

X_train = imputer_transformer.transform(X_train)
X_test = imputer_transformer.transform(X_test)

In [34]:
X_train

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,SibSp,Parch,Fare,FamilySize
331,45.500000,S,C124,False,first,male,0,0,28.5000,1
733,23.000000,S,Missing,True,second,male,0,0,13.0000,1
382,32.000000,S,Missing,True,third,male,0,0,7.9250,1
704,26.000000,S,Missing,True,third,male,1,0,7.8542,2
813,6.000000,S,Missing,True,third,female,4,2,31.2750,7
...,...,...,...,...,...,...,...,...,...,...
106,21.000000,S,Missing,True,third,female,0,0,7.6500,1
270,29.498846,S,Missing,True,first,male,0,0,31.0000,1
860,41.000000,S,Missing,True,third,male,2,0,14.1083,3
435,14.000000,S,B96 B98,False,first,female,1,2,120.0000,4


In [35]:
X_test

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,SibSp,Parch,Fare,FamilySize
709,29.498846,C,Missing,True,third,male,1,1,15.2458,3
439,31.000000,S,Missing,True,second,male,0,0,10.5000,1
840,20.000000,S,Missing,True,third,male,0,0,7.9250,1
720,6.000000,S,Missing,True,second,female,0,1,33.0000,2
39,14.000000,C,Missing,True,third,female,1,0,11.2417,2
...,...,...,...,...,...,...,...,...,...,...
433,17.000000,S,Missing,True,third,male,0,0,7.1250,1
773,29.498846,C,Missing,True,third,male,0,0,7.2250,1
25,38.000000,S,Missing,True,third,female,1,5,31.3875,7
84,17.000000,S,Missing,True,second,female,0,0,10.5000,1


In [36]:
mean_of_age = X_train['Age'].mean()
std_of_age = X_train['Age'].std()
X_train['zscore_age'] = (X_train['Age']-mean_of_age)/std_of_age

X_train = X_train[abs(X_train['zscore_age']) <= 3]
X_train.drop(['zscore_age'], axis=1, inplace=True)

C:\Users\Pikachu\AppData\Local\Temp\ipykernel_3712\4037314983.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.drop(['zscore_age'], axis=1, inplace=True)


In [37]:
fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

fare_IQR = fare_Q3 - fare_Q1

fare_minimum = fare_Q1 - 1.5*fare_IQR
fare_maximum = fare_Q3 + 1.5*fare_IQR

X_train['Fare'] = X_train['Fare'].clip(fare_minimum, fare_maximum)

C:\Users\Pikachu\AppData\Local\Temp\ipykernel_3712\1123865105.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(fare_minimum, fare_maximum)


In [38]:
X_train['Cabin_Deck'] = X_train['Cabin'].astype(str).str[0]
X_test['Cabin_Deck'] = X_test['Cabin'].astype(str).str[0]

C:\Users\Pikachu\AppData\Local\Temp\ipykernel_3712\938491672.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Cabin_Deck'] = X_train['Cabin'].astype(str).str[0]


In [39]:
X_train.drop(['Cabin'], axis=1, inplace=True)
X_train

C:\Users\Pikachu\AppData\Local\Temp\ipykernel_3712\3358116690.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.drop(['Cabin'], axis=1, inplace=True)


,Age,Embarked,missingindicator_Cabin,Pclass,Sex,SibSp,Parch,Fare,FamilySize,Cabin_Deck
331,45.500000,S,False,first,male,0,0,28.5000,1,C
733,23.000000,S,True,second,male,0,0,13.0000,1,M
382,32.000000,S,True,third,male,0,0,7.9250,1,M
704,26.000000,S,True,third,male,1,0,7.8542,2,M
813,6.000000,S,True,third,female,4,2,31.2750,7,M
...,...,...,...,...,...,...,...,...,...,...
106,21.000000,S,True,third,female,0,0,7.6500,1,M
270,29.498846,S,True,first,male,0,0,31.0000,1,M
860,41.000000,S,True,third,male,2,0,14.1083,3,M
435,14.000000,S,False,first,female,1,2,64.3625,4,B


In [40]:
encoder_scaler = ColumnTransformer(
    transformers=[
        ('pclass',OrdinalEncoder(categories=[['third', 'second','first']]),['Pclass']),
        ('embarked_sex',OneHotEncoder(sparse_output=False,drop='first'),['Embarked','Sex']),
        ('cabin_deck', OneHotEncoder(sparse_output=False, drop='first'),['Cabin_Deck']),
        ('age_scaler', StandardScaler(),['Age']),
        ('fare', MinMaxScaler(),['Fare','FamilySize']),
    ],
        remainder='passthrough',
        verbose_feature_names_out=False
)

encoder_scaler.set_output(transform='pandas')

encoder_scaler.fit(X_train)

X_train = encoder_scaler.transform(X_train)
X_test = encoder_scaler.transform(X_test)

In [41]:
X_train

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_M,Cabin_Deck_T,Age,Fare,FamilySize,missingindicator_Cabin,SibSp,Parch
331,2.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.304495,0.442804,0.0,False,0,0
733,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.495295,0.201981,0.0,True,0,0
382,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.224621,0.123131,0.0,True,0,0
704,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.255323,0.122031,0.1,True,1,0
813,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.855135,0.485920,0.6,True,4,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.655276,0.118858,0.0,True,0,0
270,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.481647,0.0,True,0,0
860,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.944537,0.219201,0.2,True,2,0
435,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.215210,1.000000,0.3,False,1,2


In [42]:
X_train.drop(['SibSp','Parch'], axis=1,inplace=True)
X_test.drop(['SibSp','Parch'], axis=1,inplace=True)

In [43]:
X_train

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_M,Cabin_Deck_T,Age,Fare,FamilySize,missingindicator_Cabin
331,2.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.304495,0.442804,0.0,False
733,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.495295,0.201981,0.0,True
382,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.224621,0.123131,0.0,True
704,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.255323,0.122031,0.1,True
813,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.855135,0.485920,0.6,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.655276,0.118858,0.0,True
270,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.481647,0.0,True
860,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.944537,0.219201,0.2,True
435,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.215210,1.000000,0.3,False
